# G1 Locomotion -- Colab Training (Phase 2)

Run cells in order. Runtime > Change runtime type > **T4 GPU** before starting.

In [13]:
from google.colab import drive
drive.mount('/content/drive')

CHECKPOINT_DIR = '/content/drive/MyDrive/g1_checkpoints'  # survives disconnects; edit if you want a different path

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [14]:
!git clone https://github.com/shubhamt2897/Humanoid_LocomotioN.git /content/g1_locomotion
%cd /content/g1_locomotion

fatal: destination path '/content/g1_locomotion' already exists and is not an empty directory.
/content/g1_locomotion


In [15]:
# Deliberately NOT installing torch -- Colab ships a CUDA-matched build preinstalled.
# requirements.txt pins torch too (for local/CPU use); skip that line here.
!pip install -q mujoco==3.12.0 rsl-rl-lib==5.5.0 tensordict==0.14.0 wandb==0.29.0 onnx==1.22.0

import torch
print('torch', torch.__version__, '| cuda available:', torch.cuda.is_available())
assert torch.cuda.is_available(), 'CUDA not available -- check Runtime > Change runtime type > T4 GPU'

torch 2.11.0+cu128 | cuda available: True


In [16]:
import wandb
wandb.login()  # paste your API key from wandb.ai/authorize when prompted, once per Colab session

True

In [17]:
RUN_NAME = 'asymmetric_payload_run'

# First run (nothing to resume from yet):
!python train.py \
  --num_envs 256 --device cuda --iterations 100 \
  --save_interval 25 \
  --wandb --run_name {RUN_NAME} \
  --log_dir {CHECKPOINT_DIR}/{RUN_NAME}

2026-08-30 13:33:05.441446: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
^C


## Resuming after a disconnect

Colab free tier can drop the session mid-run without warning. Because `--log_dir` points at
Drive, whatever was already checkpointed (every `--save_interval` iterations) is safe. To
resume: reconnect, re-run the mount/clone/install/login cells above, then run the cell below
instead of the first training cell -- pick the highest `model_<N>.pt` actually present in your
Drive checkpoint folder.

**Note:** `--iterations` here means "how many *more* iterations to run from the checkpoint",
not an absolute target -- `runner.load()` restores the saved iteration count, and `learn()` adds
`--iterations` on top of that.

In [ ]:
RUN_NAME = 'asymmetric_payload_run'
RESUME_FROM = f'{CHECKPOINT_DIR}/{RUN_NAME}/model_500.pt'  # <-- edit to the latest checkpoint you actually have

!python train.py \
  --num_envs 1024 --device cuda --iterations 1000 \
  --save_interval 25 \
  --wandb --run_name {RUN_NAME} \
  --log_dir {CHECKPOINT_DIR}/{RUN_NAME} \
  --resume {RESUME_FROM}

## Export the final policy to ONNX (also written straight to Drive)

In [ ]:
# NOTE: with --iterations 1500 (0-indexed loop), the LAST checkpoint is model_1499.pt, not
# model_1500.pt -- rsl_rl's runner saves every --save_interval iterations plus one final save
# at whatever iteration the loop actually stopped on. Check {CHECKPOINT_DIR}/{RUN_NAME}/ and use
# whichever model_<N>.pt is actually the highest N present.
!python export_policy.py \
  --checkpoint {CHECKPOINT_DIR}/{RUN_NAME}/model_1499.pt \
  --out {CHECKPOINT_DIR}/{RUN_NAME}/g1_policy.onnx